# Model Development

This notebook covers baseline model development for the Customer Churn Prediction Platform.

The preprocessing pipeline developed during Phase 6 is reused to ensure consistent feature engineering and preprocessing during model training and evaluation.

## 7.1 Baseline Model

A Logistic Regression model is established as the baseline classifier.

The baseline provides a simple, interpretable reference point against which more advanced models can be evaluated.

In [1]:
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Reusable Preprocessing Pipeline

The preprocessing workflow developed during Phase 6 is implemented as a reusable production module.

The same pipeline will be used for model training and later inference to maintain consistency between development and deployment.

In [2]:
import sys
from pathlib import Path

PROJECT_ROOT_71 = Path.cwd().parent

if str(PROJECT_ROOT_71) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT_71))

from backend.src.ml.preprocessing_pipeline import build_preprocessing_pipeline

pipeline_71 = build_preprocessing_pipeline()

print("Reusable preprocessing pipeline loaded successfully.")
print("Pipeline steps:", list(pipeline_71.named_steps.keys()))

Reusable preprocessing pipeline loaded successfully.
Pipeline steps: ['data_preparation', 'feature_engineering', 'preprocessing']


## Load Training Dataset

The raw customer churn dataset is loaded for baseline model development.

The target variable `Churn` is separated from the input features, while the reusable preprocessing pipeline handles data preparation, feature engineering, encoding, and scaling.

In [3]:
DATA_PATH_71 = "../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"

df_71 = pd.read_csv(DATA_PATH_71)

X_71 = df_71.drop(columns=["customerID", "Churn"])
y_71 = df_71["Churn"].map({
    "No": 0,
    "Yes": 1
})

print("=" * 60)
print("BASELINE DATASET")
print("=" * 60)

print("Input shape:", X_71.shape)
print("Target shape:", y_71.shape)
print("Missing target values:", y_71.isnull().sum())

BASELINE DATASET
Input shape: (7043, 19)
Target shape: (7043,)
Missing target values: 0


## Train-Test Split

An 80/20 stratified split is used for baseline model development.

The split is performed before fitting the preprocessing pipeline so that all data-dependent transformations are learned only from the training data.

In [4]:
from sklearn.model_selection import train_test_split

X_train_71, X_test_71, y_train_71, y_test_71 = train_test_split(
    X_71,
    y_71,
    test_size=0.20,
    random_state=42,
    stratify=y_71
)

print("Training data:", X_train_71.shape)
print("Testing data:", X_test_71.shape)

Training data: (5634, 19)
Testing data: (1409, 19)


## Logistic Regression Baseline

Logistic Regression is used as the baseline classifier because it is simple, interpretable, computationally efficient, and provides a strong reference point for comparing subsequent models.

In [5]:
baseline_model_71 = Pipeline(
    steps=[
        ("preprocessing", pipeline_71),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

baseline_model_71.fit(
    X_train_71,
    y_train_71
)

print("Baseline Logistic Regression trained successfully.")

Baseline Logistic Regression trained successfully.


In [6]:
y_pred_71 = baseline_model_71.predict(X_test_71)
y_prob_71 = baseline_model_71.predict_proba(X_test_71)[:, 1]

print("=" * 60)
print("BASELINE MODEL PERFORMANCE")
print("=" * 60)

print("Accuracy :", accuracy_score(y_test_71, y_pred_71))
print("Precision:", precision_score(y_test_71, y_pred_71))
print("Recall   :", recall_score(y_test_71, y_pred_71))
print("F1 Score :", f1_score(y_test_71, y_pred_71))
print("ROC-AUC  :", roc_auc_score(y_test_71, y_prob_71))

BASELINE MODEL PERFORMANCE
Accuracy : 0.7998580553584103
Precision: 0.6554054054054054
Recall   : 0.5187165775401069
F1 Score : 0.5791044776119403
ROC-AUC  : 0.8424242424242425


## 7.2 Train Multiple Models

Multiple classification algorithms are trained using the same reusable preprocessing pipeline established during Phase 6.

The following models are evaluated alongside the Logistic Regression baseline:

- Decision Tree
- Random Forest
- Linear Support Vector Machine

Using the same preprocessing workflow ensures that model comparisons are based on consistent input transformations.

In [7]:
models_72 = {
    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),
    "Linear SVM": LinearSVC(
        random_state=42,
        max_iter=5000
    )
}

trained_models_72 = {}

In [8]:
for model_name, model in models_72.items():
    model_pipeline = Pipeline(
        steps=[
            ("preprocessing", clone(pipeline_71)),
            ("classifier", model)
        ]
    )

    model_pipeline.fit(
        X_train_71,
        y_train_71
    )

    trained_models_72[model_name] = model_pipeline

    print(f"{model_name} trained successfully.")

Decision Tree trained successfully.
Random Forest trained successfully.
Linear SVM trained successfully.


In [9]:
print("=" * 60)
print("MODEL TRAINING VERIFICATION")
print("=" * 60)

print("Models trained:", len(trained_models_72))

for model_name in trained_models_72:
    print("-", model_name)

MODEL TRAINING VERIFICATION
Models trained: 3
- Decision Tree
- Random Forest
- Linear SVM


## 7.3 Hyperparameter Tuning

Hyperparameter tuning is performed to identify improved configurations for the tree-based candidate models.

Randomized or exhaustive search is limited to a small, portfolio-friendly parameter space to balance model improvement with computational efficiency.

The tuning process uses cross-validation on the training data only.

In [10]:
from sklearn.model_selection import GridSearchCV

In [11]:
decision_tree_pipeline_73 = Pipeline(
    steps=[
        ("preprocessing", clone(pipeline_71)),
        ("classifier", DecisionTreeClassifier(random_state=42))
    ]
)

random_forest_pipeline_73 = Pipeline(
    steps=[
        ("preprocessing", clone(pipeline_71)),
        (
            "classifier",
            RandomForestClassifier(
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

decision_tree_params_73 = {
    "classifier__max_depth": [3, 5, 8, None],
    "classifier__min_samples_split": [2, 5, 10]
}

random_forest_params_73 = {
    "classifier__n_estimators": [100, 200],
    "classifier__max_depth": [None, 8, 12],
    "classifier__min_samples_split": [2, 5]
}

In [12]:
decision_tree_search_73 = GridSearchCV(
    estimator=decision_tree_pipeline_73,
    param_grid=decision_tree_params_73,
    scoring="roc_auc",
    cv=3,
    n_jobs=-1
)

random_forest_search_73 = GridSearchCV(
    estimator=random_forest_pipeline_73,
    param_grid=random_forest_params_73,
    scoring="roc_auc",
    cv=3,
    n_jobs=-1
)

In [13]:
decision_tree_search_73.fit(
    X_train_71,
    y_train_71
)

random_forest_search_73.fit(
    X_train_71,
    y_train_71
)

print("Decision Tree tuning completed.")
print("Random Forest tuning completed.")

Decision Tree tuning completed.
Random Forest tuning completed.


In [14]:
print("=" * 60)
print("HYPERPARAMETER TUNING RESULTS")
print("=" * 60)

print("Decision Tree best parameters:")
print(decision_tree_search_73.best_params_)

print("\nDecision Tree best CV ROC-AUC:")
print(round(decision_tree_search_73.best_score_, 4))

print("\nRandom Forest best parameters:")
print(random_forest_search_73.best_params_)

print("\nRandom Forest best CV ROC-AUC:")
print(round(random_forest_search_73.best_score_, 4))

HYPERPARAMETER TUNING RESULTS
Decision Tree best parameters:
{'classifier__max_depth': 5, 'classifier__min_samples_split': 10}

Decision Tree best CV ROC-AUC:
0.8236

Random Forest best parameters:
{'classifier__max_depth': 8, 'classifier__min_samples_split': 5, 'classifier__n_estimators': 200}

Random Forest best CV ROC-AUC:
0.8449


## 7.4 Cross-Validation

Cross-validation is performed to evaluate the stability of the candidate models across multiple training and validation splits.

A 5-fold stratified cross-validation strategy is used with ROC-AUC as the evaluation metric.

The evaluation includes:

- Logistic Regression baseline
- Tuned Decision Tree
- Tuned Random Forest
- Linear SVM

Cross-validation is performed only on the training dataset. The held-out test dataset remains untouched for final evaluation.

In [15]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

In [16]:
cv_74 = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_models_74 = {
    "Logistic Regression": baseline_model_71,
    "Tuned Decision Tree": decision_tree_search_73.best_estimator_,
    "Tuned Random Forest": random_forest_search_73.best_estimator_,
    "Linear SVM": trained_models_72["Linear SVM"]
}

cv_results_74 = {}

for model_name, model in cv_models_74.items():
    scores = cross_val_score(
        model,
        X_train_71,
        y_train_71,
        cv=cv_74,
        scoring="roc_auc",
        n_jobs=-1
    )

    cv_results_74[model_name] = scores

    print(
        f"{model_name}: "
        f"Mean ROC-AUC = {scores.mean():.4f}, "
        f"Std = {scores.std():.4f}"
    )

Logistic Regression: Mean ROC-AUC = 0.8465, Std = 0.0117
Tuned Decision Tree: Mean ROC-AUC = 0.8292, Std = 0.0107
Tuned Random Forest: Mean ROC-AUC = 0.8450, Std = 0.0120
Linear SVM: Mean ROC-AUC = 0.8432, Std = 0.0110


In [17]:
print("=" * 60)
print("CROSS-VALIDATION SUMMARY")
print("=" * 60)

for model_name, scores in cv_results_74.items():
    print(
        f"{model_name:<22} "
        f"Mean: {scores.mean():.4f} | "
        f"Std: {scores.std():.4f}"
    )

CROSS-VALIDATION SUMMARY
Logistic Regression    Mean: 0.8465 | Std: 0.0117
Tuned Decision Tree    Mean: 0.8292 | Std: 0.0107
Tuned Random Forest    Mean: 0.8450 | Std: 0.0120
Linear SVM             Mean: 0.8432 | Std: 0.0110
